In [27]:
import os
DATA_PATH = '/kaggle/input/competitions/home-credit-default-risk'
# check the data file name
print(os.listdir(DATA_PATH))

['sample_submission.csv', 'bureau_balance.csv', 'POS_CASH_balance.csv', 'application_train.csv', 'HomeCredit_columns_description.csv', 'application_test.csv', 'previous_application.csv', 'credit_card_balance.csv', 'installments_payments.csv', 'bureau.csv']


In [28]:
# processing the main table
import pandas as pd
import numpy as np

def get_application_data():
    """
    data load and data clean of main table
    """
    # 1. data load
    train = pd.read_csv(os.path.join(DATA_PATH, 'application_train.csv'))
    test = pd.read_csv(os.path.join(DATA_PATH, 'application_test.csv'))
    
    # merge the train data and test table, to ensure the features same
    df = pd.concat([train, test], axis=0, ignore_index=True)
    print(f"Raw data read complete, shape is: {df.shape}")

    # 2. process the outliers
    # employed outliers
    df['DAYS_EMPLOYED_ANOM'] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
    df['DAYS_EMPLOYED'].replace({365243: np.nan}, inplace=True)
    
    # convert negative days into positive years
    df['DAYS_BIRTH'] = abs(df['DAYS_BIRTH']) / 365
    df['DAYS_EMPLOYED'] = abs(df['DAYS_EMPLOYED']) / 365
    df['DAYS_REGISTRATION'] = abs(df['DAYS_REGISTRATION']) / 365
    df['DAYS_ID_PUBLISH'] = abs(df['DAYS_ID_PUBLISH']) / 365

    # 3. calculate the domain knowledge features
    df['PAYMENT_RATE'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']         # Loan Interest Rate/Repayment Rate
    df['CREDIT_INCOME_PERCENT'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL'] # Loan-to-deposit ratio
    df['ANNUITY_INCOME_PERCENT'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL'] # Annuity Income Ratio
    df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS'] # Per capita income

    # 4. Label Encoding
    # For fields with only two categories, perform direct conversion.
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    for col in df.columns:
        if df[col].dtype == 'object':
            if len(list(df[col].unique())) <= 2:
                df[col] = le.fit_transform(df[col].astype(str))

    # 5. One-Hot Encoding (for multi-class fields)
    df = pd.get_dummies(df)

    print(f"Preprocessing completed. Current feature dimension: {df.shape}")
    return df

app_data = get_application_data() # app_data is clean data for next steps

Raw data read complete, shape is: (356255, 122)


/tmp/ipykernel_55/1542614413.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['DAYS_EMPLOYED'].replace({365243: np.nan}, inplace=True)


Preprocessing completed. Current feature dimension: (356255, 248)


In [29]:
def process_bureau_and_balance():
    # 1. load data
    bureau = pd.read_csv(os.path.join(DATA_PATH, 'bureau.csv'))
    bb = pd.read_csv(os.path.join(DATA_PATH, 'bureau_balance.csv'))

    # process the bureau_balance
    # One-hot encoding
    bb_cats = pd.get_dummies(bb.select_dtypes(include=['object']), prefix='bb')
    bb_cats['SK_ID_BUREAU'] = bb['SK_ID_BUREAU']
    
    # aggregate the status information in bb
    bb_agg = bb_cats.groupby('SK_ID_BUREAU').mean() # calculate the frequency of occurrence for each state
    
    # process the bureau
    # merge bb's aggregated features into bureau
    bureau = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')
    
    # perform one-hot encoding on the categorical variable “bureau”
    bureau_cats = pd.get_dummies(bureau.select_dtypes(include=['object']), prefix='bu')
    bureau_numeric = bureau.select_dtypes(include=['number'])
    bureau_cats['SK_ID_CURR'] = bureau['SK_ID_CURR']
    bureau_numeric['SK_ID_CURR'] = bureau['SK_ID_CURR']

    # numeric feature aggregation
    bureau_num_agg = bureau_numeric.groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'sum'])
    bureau_num_agg.columns = [f'BU_{c[0]}_{c[1].upper()}' for c in bureau_num_agg.columns]
    
    # category-based feature aggregation: calculate the mean (percentage)
    bureau_cat_agg = bureau_cats.groupby('SK_ID_CURR').mean()
    bureau_cat_agg.columns = [f'BU_{c}_MEAN' for c in bureau_cat_agg.columns]

    # merge the result
    bureau_final = bureau_num_agg.merge(bureau_cat_agg, on='SK_ID_CURR', how='left')
    
    print(f"Bureau Feature extraction completed. Number of new features added: {bureau_final.shape[1]}")
    return bureau_final

# execute merge process
bureau_features = process_bureau_and_balance()

# Link back to the main table 
app_data = app_data.merge(bureau_features, on='SK_ID_CURR', how='left')
print(f"Total number of features in the current table: {app_data.shape[1]}")

Bureau Feature extraction completed. Number of new features added: 107
Total number of features in the current table: 355


In [30]:
# check app_data
display(app_data.head(10))

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,BU_bu_Interbank credit_MEAN,BU_bu_Loan for business development_MEAN,BU_bu_Loan for purchase of shares (margin lending)_MEAN,BU_bu_Loan for the purchase of equipment_MEAN,BU_bu_Loan for working capital replenishment_MEAN,BU_bu_Microloan_MEAN,BU_bu_Mobile operator loan_MEAN,BU_bu_Mortgage_MEAN,BU_bu_Real estate loan_MEAN,BU_bu_Unknown type of loan_MEAN
0,100002,1.0,0,0,1,0,202500.0,406597.5,24700.5,351000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,100003,0.0,0,0,0,0,270000.0,1293502.5,35698.5,1129500.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0.0,1,1,1,0,67500.0,135000.0,6750.0,135000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0.0,0,0,1,0,135000.0,312682.5,29686.5,297000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0.0,0,0,1,0,121500.0,513000.0,21865.5,513000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,100008,0.0,0,0,1,0,99000.0,490495.5,27517.5,454500.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,100009,0.0,0,1,1,1,171000.0,1560726.0,41301.0,1395000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,100010,0.0,0,1,1,0,360000.0,1530000.0,42075.0,1530000.0,...,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,100011,0.0,0,0,1,0,112500.0,1019610.0,33826.5,913500.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,100012,0.0,1,0,1,0,135000.0,405000.0,20250.0,405000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
# view total number of rows, total number of columns, and total memory usage
app_data.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 356255 entries, 0 to 356254
Data columns (total 355 columns):
 #    Column                                                   Non-Null Count   Dtype  
---   ------                                                   --------------   -----  
 0    SK_ID_CURR                                               356255 non-null  int64  
 1    TARGET                                                   307511 non-null  float64
 2    NAME_CONTRACT_TYPE                                       356255 non-null  int64  
 3    FLAG_OWN_CAR                                             356255 non-null  int64  
 4    FLAG_OWN_REALTY                                          356255 non-null  int64  
 5    CNT_CHILDREN                                             356255 non-null  int64  
 6    AMT_INCOME_TOTAL                                         356255 non-null  float64
 7    AMT_CREDIT                                               356255 non-null  float64
 8    AM

In [32]:
# the 10 features with the highest missing rates
missing_stats = app_data.isnull().mean().sort_values(ascending=False)
print(missing_stats.head(10))

COMMONAREA_AVG              0.697141
COMMONAREA_MODE             0.697141
COMMONAREA_MEDI             0.697141
NONLIVINGAPARTMENTS_MODE    0.692933
NONLIVINGAPARTMENTS_AVG     0.692933
NONLIVINGAPARTMENTS_MEDI    0.692933
LIVINGAPARTMENTS_AVG        0.682037
LIVINGAPARTMENTS_MODE       0.682037
LIVINGAPARTMENTS_MEDI       0.682037
FLOORSMIN_AVG               0.676785
dtype: float64


In [33]:
# Optimize data to free up unnecessary memory
import gc


def reduce_mem_usage(df):
    """ 
    Iterate through all columns of a dataframe and modify the data type
    to reduce memory consumption based on the range of values.
    """
    for col in df.columns:
        col_type = df[col].dtype
        
        # Only process numerical columns (exclude objects/strings)
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            
            # Handling Integer types
            if str(col_type)[:3] == 'int':
                # Downcast to the smallest possible integer type
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            # Handling Float types
            else:
                # Downcast float64 to float32 to save 50% memory with minimal precision loss
                df[col] = df[col].astype(np.float32)
                
    return df

def process_previous_applications():
    print("Loading and processing Previous Applications...")
    # load raw data from the specified path
    prev = pd.read_csv(os.path.join(DATA_PATH, 'previous_application.csv'))
    
    # 1. handle anomalous values
    # 365243 represents 'Not Applicable' or 'Unknown' in time-related columns
    time_cols = ['DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION']
    for col in time_cols:
        prev[col] = prev[col].replace(365243, np.nan)
    
    # 2. simple feature engineering
    # oercentage of the amount actually credited vs. the amount applied for
    prev['APP_CREDIT_PERC'] = prev['AMT_APPLICATION'] / prev['AMT_CREDIT']
    
    # 3. One-Hot
    # We include dummy_na=True because missing values in credit history often carry significant risk signals
    prev_cat_cols = [col for col in prev.columns if prev[col].dtype == 'object']
    prev = pd.get_dummies(prev, columns=prev_cat_cols, dummy_na=True)
    
    # 4. define aggregation logic
    # numerical aggregations: capturing stats for financial amounts and decision timing
    num_aggregations = {
        'AMT_ANNUITY': ['max', 'mean', 'sum'],
        'AMT_APPLICATION': ['max', 'mean', 'sum'],
        'AMT_CREDIT': ['max', 'mean', 'sum'],
        'APP_CREDIT_PERC': ['max', 'mean', 'var'],
        'AMT_DOWN_PAYMENT': ['max', 'mean'],
        'AMT_GOODS_PRICE': ['max', 'mean', 'sum'],
        'DAYS_DECISION': ['max', 'mean', 'min'],
        'CNT_PAYMENT': ['mean', 'sum'],
    }
    
    # categorical aggregations: capturing the frequency/ratio of different application statuses
    cat_aggregations = {}
    for col in [c for c in prev.columns if 'NAME_' in c or 'CODE_' in c]:
        cat_aggregations[col] = ['mean']
    
    # 5. execute grouping and aggregation
    # convert many-to-one relationship into a single row per SK_ID_CURR
    prev_agg = prev.groupby('SK_ID_CURR').agg({**num_aggregations, **cat_aggregations})
    
    # standardize column naming convention: Prefix_FeatureName_StatMethod
    prev_agg.columns = pd.Index(['PREV_' + e[0] + "_" + e[1].upper() for e in prev_agg.columns.tolist()])
    
    # 6. optimized join for feature count
    # use .join() instead of direct assignment to avoid DataFrame fragmentation
    prev_count = prev.groupby('SK_ID_CURR').size().to_frame('PREV_COUNT')
    prev_agg = prev_agg.join(prev_count, how='left')
    
    # create a deep copy to ensure a contiguous memory layout 
    prev_agg = prev_agg.copy()
    
    # 7. apply memory optimization
    prev_agg = reduce_mem_usage(prev_agg)
    
    # clear memory explicitly
    del prev
    gc.collect()
    
    return prev_agg

# execute
prev_features = process_previous_applications()
app_data = app_data.merge(prev_features, on='SK_ID_CURR', how='left')

# final memory optimization for the combined master table
app_data = reduce_mem_usage(app_data)
print(f"Processing Complete! Current combined table shape: {app_data.shape}")

Loading and processing Previous Applications...
Processing Complete! Current combined table shape: (356255, 499)


In [34]:
# --- 1. memory optimization ---
def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.float32)
    return df

# 2. Installments Payments Processing Function
def process_installments_payments():
    print("Loading and processing Installments Payments...")
    ins = pd.read_csv(os.path.join(DATA_PATH, 'installments_payments.csv'))
    
    # Basic Derived Features
    # 1. Payment amount differences and percentages
    ins['PAYMENT_PERC'] = ins['AMT_PAYMENT'] / ins['AMT_INSTALMENT']
    ins['PAYMENT_DIFF'] = ins['AMT_INSTALMENT'] - ins['AMT_PAYMENT']
    
    # 2. Days Past Due (DPD) and Days Before Due (DBD)
    ins['DPD'] = ins['DAYS_ENTRY_PAYMENT'] - ins['DAYS_INSTALMENT']
    ins['DBD'] = ins['DAYS_INSTALMENT'] - ins['DAYS_ENTRY_PAYMENT']
    # Only values > 0 are logically meaningful for these signals; otherwise, set to 0
    ins['DPD'] = ins['DPD'].apply(lambda x: x if x > 0 else 0)
    ins['DBD'] = ins['DBD'].apply(lambda x: x if x > 0 else 0)
    
    # Aggregation Logic
    aggregations = {
        'NUM_INSTALMENT_VERSION': ['nunique'],
        'DPD': ['max', 'mean', 'sum'],
        'DBD': ['max', 'mean', 'sum'],
        'PAYMENT_PERC': ['max', 'mean', 'sum', 'var'],
        'PAYMENT_DIFF': ['max', 'mean', 'sum', 'var'],
        'AMT_INSTALMENT': ['max', 'mean', 'sum'],
        'AMT_PAYMENT': ['min', 'max', 'mean', 'sum'],
        'DAYS_ENTRY_PAYMENT': ['max', 'mean', 'sum']
    }
    
    # Execute GroupBy aggregation
    ins_agg = ins.groupby('SK_ID_CURR').agg(aggregations)
    ins_agg.columns = pd.Index(['INS_' + e[0] + "_" + e[1].upper() for e in ins_agg.columns.tolist()])
    
    # Calculate the total number of installment records per client
    ins_count = ins.groupby('SK_ID_CURR').size().to_frame('INS_COUNT')
    ins_agg = ins_agg.join(ins_count, how='left')
    
    # De-fragment the DataFrame and perform memory optimization
    ins_agg = ins_agg.copy()
    ins_agg = reduce_mem_usage(ins_agg)
    
    print(f"Installments Table processing complete. New features added: {ins_agg.shape[1]}")
    
    del ins
    gc.collect()
    return ins_agg

# 3.Execute Merging Process
ins_features = process_installments_payments()
app_data = app_data.merge(ins_features, on='SK_ID_CURR', how='left')

# Final global memory optimization for the entire dataset
app_data = reduce_mem_usage(app_data)
print(f"Final wide table dimensions: {app_data.shape}")

Loading and processing Installments Payments...
Installments Table processing complete. New features added: 26
Final wide table dimensions: (356255, 525)
